[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/B.%20Warehouse%20%26%20Fulfillment%20Center%20Operations%20%28Inside%20the%20Four%20Walls%29/061.%20The%20Order%20Batching%20Problem/P61-Tier-2_executed.ipynb)

---


# 61. The Order Batching Problem
## Tier 2: Constraint-Propagation Backtracking

### Purpose and scope

Given a set of customer orders, assign every order to exactly one batch. Each batch has a capacity limit. The search uses recursive backtracking: it assigns orders one at a time, rejects assignments that exceed capacity, and records the least-cost complete assignment found.

The batch cost is estimated by a **nearest-neighbor route heuristic**. Starting at a depot at $(0,0)$, the heuristic repeatedly visits the nearest unvisited order location and then returns to the depot. The backtracking search is exhaustive unless its time limit is reached, so it can find the minimum batching cost **for this specified route-cost estimate**. It does not prove that the estimated routes are globally shortest physical routes.

### Modeling assumptions

- Order volumes are additive, and a batch is feasible when its total volume is no greater than the picker capacity.
- There is no limit on the number of batches and no fixed cost for opening a batch.
- A batch's route starts and ends at the depot; distances are Euclidean in the coordinate system used by the example.
- Order locations are represented by two-dimensional coordinates. Travel times, aisle networks, precedence rules, time windows, and interactions between pickers are outside this model.
- Orders are processed in their given list order. Batch labels are created canonically: the next new batch receives the next integer label.

### Core concepts

A **search state** is a partial assignment of the orders already considered. **Backtracking** explores a possible assignment, recursively considers later orders, and then removes the assignment so another choice can be explored. **Constraint propagation**, also called forward checking here, inspects unassigned orders after a tentative assignment and rejects the branch if any remaining order cannot fit in an existing batch or a new batch. With unlimited batches, a new batch is always available for any order whose volume does not exceed capacity; consequently, this forward check is weak for feasible instances. Capacity checks on each attempted assignment still prevent infeasible branches.

The search is a depth-first traversal of a tree. At depth $k$, the first $k$ orders have been assigned. The recursion stops at depth $n$, when all $n$ orders have assignments. Each complete assignment is evaluated, and the best cost found is retained.

### Mathematical model and search recurrence

Let $I=\{0,\ldots,n-1\}$ be the order indices, let $v_i$ be the volume of order $i$, and let $Q$ be batch capacity. A complete assignment is a function $a:I\to\mathbb{N}_0$, where $a(i)$ is the batch assigned to order $i$. For each batch label $b$, capacity requires

$$
\sum_{i\in I:a(i)=b} v_i \leq Q.
$$

Let $C(S)$ denote the nearest-neighbor route cost of the orders in batch $S$, including travel from the depot and the return trip. The objective evaluated by the search is

$$
\min_{a}\; \sum_{b\in a(I)} C\bigl(\{i\in I:a(i)=b\}\bigr)
\quad\text{subject to the batch-capacity constraints.}
$$

For a partial assignment $A$ of the first $k$ orders, let $F(A,k)$ be the set of capacity-feasible next choices: assignment to an existing batch with enough remaining capacity, or assignment to a new batch. The recursive search is

$$
\operatorname{Search}(A,k)=
\begin{cases}
\text{evaluate } A \text{ and update the incumbent}, & k=n,\\
\displaystyle\bigcup_{b\in F(A,k)}\operatorname{Search}(A\cup\{k\mapsto b\},k+1), & k<n.
\end{cases}
$$

The empty assignment at $k=0$ is the base state from which recursion begins. The implementation stores the partial assignment in a dictionary, uses `level` for $k$, and evaluates complete assignments with `evaluate_solution`. Constraint propagation is a feasibility check on the remaining orders; it does not provide a lower bound on the objective, so the algorithm does not prune a branch merely because its current cost seems high.

At each depth, the number of possible next choices can grow with the number of batches already created. The number of canonical ways to partition $n$ orders is the Bell number $B_n$, which grows super-polynomially. Thus, exhaustive search is practical only for relatively small instances. Depth-first search stores a path of assignments, requiring $O(n)$ assignment space apart from the distance matrix and recursion overhead.

### Learning sequence

1. Define orders, capacity, locations, and the batch route-cost estimate.
2. Build a recursive backtracking solver with capacity checks and forward checking.
3. Run the solver on the six-order instance and inspect the resulting batches.
4. Compare its search with a small-instance exhaustive enumeration baseline.
5. Trace early search levels, examine scaling, and interpret the visualizations.
6. Review the algorithm's complexity, guarantees, and modeling limits.

In [ ]:
import time
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Keep the random-number stream reproducible for experiments that use NumPy.
np.random.seed(42)

print("Order Batching: Constraint-Propagation Backtracking")
print("Required packages imported successfully.")

In [ ]:
class Order:
    """A customer order with a volume and a two-dimensional location."""

    def __init__(self, id, volume, location):
        self.id = id
        self.volume = volume
        self.location = location

    def __repr__(self):
        return f"Order({self.id}, volume={self.volume}, location={self.location})"


class BacktrackingSolver:
    """Depth-first batching search with capacity checks and forward checking."""

    def __init__(self, orders, capacity):
        self.orders = orders
        self.capacity = capacity
        self.num_orders = len(orders)

        # These counters describe states visited and branches rejected by the search.
        self.nodes_explored = 0
        self.pruned_nodes = 0
        self.backtrack_calls = 0
        self.propagation_calls = 0

        self.best_solution = None
        self.best_cost = float("inf")
        self.timed_out = False
        self.distances = self._calculate_distances()

    def _calculate_distances(self):
        """Return the pairwise Euclidean distances between order locations."""
        distances = np.zeros((self.num_orders, self.num_orders))

        for i in range(self.num_orders):
            for j in range(self.num_orders):
                location_i = self.orders[i].location
                location_j = self.orders[j].location
                distances[i, j] = np.sqrt(
                    (location_i[0] - location_j[0]) ** 2
                    + (location_i[1] - location_j[1]) ** 2
                )

        return distances

    def calculate_batch_cost(self, batch_orders):
        """Estimate a batch's depot-to-depot travel distance by nearest neighbor."""
        if not batch_orders:
            return 0.0

        depot = (0, 0)
        locations = [depot] + [self.orders[i].location for i in batch_orders]
        unvisited = set(range(1, len(locations)))
        current = 0
        total_cost = 0.0

        # Visiting a single order still requires traveling from and back to the depot.
        while unvisited:
            nearest = min(
                unvisited,
                key=lambda candidate: np.sqrt(
                    (locations[current][0] - locations[candidate][0]) ** 2
                    + (locations[current][1] - locations[candidate][1]) ** 2
                ),
            )
            total_cost += np.sqrt(
                (locations[current][0] - locations[nearest][0]) ** 2
                + (locations[current][1] - locations[nearest][1]) ** 2
            )
            current = nearest
            unvisited.remove(nearest)

        total_cost += np.sqrt(
            (locations[current][0] - depot[0]) ** 2
            + (locations[current][1] - depot[1]) ** 2
        )
        return total_cost

    def evaluate_solution(self, assignment):
        """Sum the estimated route costs over all batches in a complete assignment."""
        batches = defaultdict(list)
        for order_index, batch_id in assignment.items():
            batches[batch_id].append(order_index)

        return sum(
            self.calculate_batch_cost(batch_orders)
            for batch_orders in batches.values()
        )

    def propagate_constraints(self, assignment, remaining_orders):
        """Forward-check whether every remaining order has some feasible batch."""
        self.propagation_calls += 1
        batch_loads = defaultdict(int)

        for order_index, batch_id in assignment.items():
            batch_loads[batch_id] += self.orders[order_index].volume

        for order_index in remaining_orders:
            order_volume = self.orders[order_index].volume
            fits_existing_batch = any(
                load + order_volume <= self.capacity
                for load in batch_loads.values()
            )
            fits_new_batch = order_volume <= self.capacity

            if not fits_existing_batch and not fits_new_batch:
                self.pruned_nodes += 1
                return False

        return True

    def backtrack_search(self, assignment, level, start_time, max_time=30):
        """Explore feasible assignments recursively and update the best solution."""
        if time.time() - start_time >= max_time:
            self.timed_out = True
            return

        self.backtrack_calls += 1
        self.nodes_explored += 1

        if level == self.num_orders:
            cost = self.evaluate_solution(assignment)
            if cost < self.best_cost:
                self.best_cost = cost
                self.best_solution = assignment.copy()
            return

        current_order = self.orders[level]
        existing_batches = sorted(set(assignment.values()))

        # Existing batches are tried first; infeasible choices are rejected immediately.
        for batch_id in existing_batches:
            batch_load = sum(
                self.orders[index].volume
                for index, assigned_batch in assignment.items()
                if assigned_batch == batch_id
            )
            if batch_load + current_order.volume > self.capacity:
                self.pruned_nodes += 1
                continue

            assignment[level] = batch_id
            remaining_orders = range(level + 1, self.num_orders)
            if self.propagate_constraints(assignment, remaining_orders):
                self.backtrack_search(
                    assignment, level + 1, start_time, max_time
                )
            del assignment[level]

            if self.timed_out:
                return

        # Canonical labels avoid exploring equivalent permutations of batch names.
        new_batch_id = max(existing_batches) + 1 if existing_batches else 0
        if current_order.volume > self.capacity:
            self.pruned_nodes += 1
            return

        assignment[level] = new_batch_id
        remaining_orders = range(level + 1, self.num_orders)
        if self.propagate_constraints(assignment, remaining_orders):
            self.backtrack_search(assignment, level + 1, start_time, max_time)
        del assignment[level]

    def solve(self, max_time=30):
        """Run the search and report whether the search completed within its limit."""
        if max_time <= 0:
            raise ValueError("max_time must be positive.")

        print(f"Starting backtracking search for {self.num_orders} orders...")
        self.nodes_explored = 0
        self.pruned_nodes = 0
        self.backtrack_calls = 0
        self.propagation_calls = 0
        self.best_solution = None
        self.best_cost = float("inf")
        self.timed_out = False

        start_time = time.time()
        self.backtrack_search({}, 0, start_time, max_time)
        search_time = time.time() - start_time

        counted_search_events = self.nodes_explored + self.pruned_nodes
        prune_percentage = (
            100 * self.pruned_nodes / counted_search_events
            if counted_search_events
            else 0.0
        )

        return {
            "solution": self.best_solution,
            "cost": self.best_cost,
            "search_time": search_time,
            "nodes_explored": self.nodes_explored,
            "pruned_nodes": self.pruned_nodes,
            "backtrack_calls": self.backtrack_calls,
            "propagation_calls": self.propagation_calls,
            "prune_percentage": prune_percentage,
            "status": "time limit reached" if self.timed_out else "complete",
        }


print("Backtracking solver defined successfully.")

## 1. Define the order-batching instance

Each `Order` has a volume and a coordinate. The solver uses list positions (`0` through `n - 1`) as order indices in assignments and in the distance matrix. The capacity restriction is applied to the sum of order volumes in each batch.

In [ ]:
# Six orders, with the example's volumes, locations, and capacity.
orders_example = [
    Order(0, 3, (2, 8)),
    Order(1, 2, (5, 3)),
    Order(2, 4, (8, 12)),
    Order(3, 1, (3, 6)),
    Order(4, 3, (9, 4)),
    Order(5, 2, (6, 10)),
]

solver = BacktrackingSolver(orders_example, capacity=7)

print("Problem instance")
print(f"Number of orders: {len(orders_example)}")
print(f"Order volumes: {[order.volume for order in orders_example]}")
print(f"Picker capacity: {solver.capacity}")
print("\nOrder details:")
for order in orders_example:
    print(
        f"  Order {order.id}: volume={order.volume}, "
        f"location={order.location}"
    )

print("\nPairwise Euclidean distance matrix:")
dist_df = pd.DataFrame(
    solver.distances,
    index=[f"O{index}" for index in range(len(orders_example))],
    columns=[f"O{index}" for index in range(len(orders_example))],
)
print(dist_df.round(2))

## 2. Run the backtracking search

At each level, the solver first tries existing batches that have enough unused capacity, then considers one new batch. A tentative assignment is checked against the remaining orders before recursion continues. At a leaf of the search tree, `evaluate_solution` sums the nearest-neighbor route estimates for all batches.

The reported pruning percentage is an operational counter: rejected branches divided by the sum of explored states and rejected branches. It is not the percentage of every mathematically possible assignment eliminated. A time-limited run reports that status explicitly; a best solution found before the limit is not necessarily proven optimal.

In [ ]:
print("Backtracking search with constraint propagation")
results = solver.solve(max_time=30)

print("\nSearch results")
print(f"Status: {results['status']}")
if results["solution"] is not None:
    print(f"Best estimated travel cost: {results['cost']:.2f}")
else:
    print("No feasible complete assignment was found.")
print(f"Search time: {results['search_time']:.4f} seconds")
print(f"Search states explored: {results['nodes_explored']:,}")
print(f"Branches rejected: {results['pruned_nodes']:,}")
print(f"Operational rejection rate: {results['prune_percentage']:.1f}%")
print(f"Backtracking calls: {results['backtrack_calls']:,}")
print(f"Forward-check calls: {results['propagation_calls']:,}")

if results["solution"] is not None:
    batches = defaultdict(list)
    for order_index, batch_id in results["solution"].items():
        batches[batch_id].append(order_index)

    print("\nBest batching found:")
    displayed_cost = 0.0
    for batch_id, order_indices in sorted(batches.items()):
        batch_volume = sum(orders_example[i].volume for i in order_indices)
        batch_cost = solver.calculate_batch_cost(order_indices)
        displayed_cost += batch_cost
        print(
            f"  Batch {batch_id}: orders {order_indices}, "
            f"volume {batch_volume}/{solver.capacity}, "
            f"estimated route cost {batch_cost:.2f}"
        )

    assert set(results["solution"]) == set(range(len(orders_example)))
    assert all(
        sum(orders_example[i].volume for i in order_indices) <= solver.capacity
        for order_indices in batches.values()
    )
    assert np.isclose(displayed_cost, results["cost"])
    if results["status"] == "complete":
        print("The search completed; the reported cost is optimal for this model.")
    else:
        print("The search stopped at its time limit; optimality is not established.")

## 3. Compare with naive enumeration

The baseline recursively generates canonical batch assignments without checking capacity until a complete assignment is reached. It is included as a small-instance reference for the backtracking search, not as a scalable approach. Both methods evaluate complete feasible assignments with the same nearest-neighbor cost function. Their event counts are different measures—search states versus feasible complete assignments—so those counts should not be treated as a direct percentage reduction in a common search space. Runtime comparisons are also sensitive to machine load and should be interpreted cautiously.

In [ ]:
def naive_enumeration_solver(orders, capacity, max_time=30):
    """Enumerate canonical assignments and check capacity at complete leaves."""
    n = len(orders)
    best_cost = float("inf")
    best_solution = None
    solutions_checked = 0
    timed_out = False
    start_time = time.time()
    temp_solver = BacktrackingSolver(orders, capacity)

    def generate_assignments_recursive(assignment, order_index):
        nonlocal best_cost, best_solution, solutions_checked, timed_out

        if time.time() - start_time >= max_time:
            timed_out = True
            return

        if order_index == n:
            batches = defaultdict(list)
            for assigned_order, batch_id in assignment.items():
                batches[batch_id].append(assigned_order)

            is_feasible = all(
                sum(orders[i].volume for i in batch_orders) <= capacity
                for batch_orders in batches.values()
            )
            if is_feasible:
                cost = temp_solver.evaluate_solution(assignment)
                solutions_checked += 1
                if cost < best_cost:
                    best_cost = cost
                    best_solution = assignment.copy()
            return

        existing_batches = sorted(set(assignment.values()))
        for batch_id in existing_batches:
            assignment[order_index] = batch_id
            generate_assignments_recursive(assignment, order_index + 1)
            del assignment[order_index]
            if timed_out:
                return

        new_batch_id = max(existing_batches) + 1 if existing_batches else 0
        assignment[order_index] = new_batch_id
        generate_assignments_recursive(assignment, order_index + 1)
        del assignment[order_index]

    if max_time <= 0:
        raise ValueError("max_time must be positive.")

    generate_assignments_recursive({}, 0)
    return {
        "solution": best_solution,
        "cost": best_cost,
        "solutions_checked": solutions_checked,
        "time": time.time() - start_time,
        "status": "time limit reached" if timed_out else "complete",
    }


print("Comparison with naive enumeration")
naive_results = naive_enumeration_solver(
    orders_example, capacity=7, max_time=10
)
print(f"Naive enumeration status: {naive_results['status']}")
if naive_results["solution"] is not None:
    print(f"  Best estimated travel cost: {naive_results['cost']:.2f}")
else:
    print("  No feasible complete assignment was found.")
print(f"  Feasible complete assignments evaluated: {naive_results['solutions_checked']:,}")
print(f"  Elapsed time: {naive_results['time']:.4f} seconds")

print("\nBacktracking with forward checking")
print(f"  Search status: {results['status']}")
if results["solution"] is not None:
    print(f"  Best estimated travel cost: {results['cost']:.2f}")
print(f"  Search states explored: {results['nodes_explored']:,}")
print(f"  Branches rejected: {results['pruned_nodes']:,}")
print(f"  Elapsed time: {results['search_time']:.4f} seconds")

if (
    naive_results["time"] > 0
    and results["search_time"] > 0
    and naive_results["status"] == "complete"
    and results["status"] == "complete"
):
    speedup = naive_results["time"] / results["search_time"]
    print(f"\nObserved runtime ratio (naive / backtracking): {speedup:.2f}")
    print("This ratio is instance- and machine-dependent.")

## 4. Trace the early search tree

This trace illustrates the order in which assignments are considered. It is a demonstration of branching and capacity checks, not a second optimization run. The displayed levels correspond to order indices. A branch can be rejected because an order exceeds capacity in a proposed batch, or because forward checking finds a remaining order with no feasible batch.

In [ ]:
def demonstrate_search_tree(solver, max_depth=4):
    """Print the explored assignment choices through a limited search depth."""
    print(f"Search-tree trace: first {max_depth} levels")

    def trace_search(assignment, level):
        if level >= max_depth or level >= len(solver.orders):
            return

        current_order = solver.orders[level]
        existing_batches = sorted(set(assignment.values()))
        print(
            f"\nLevel {level}: assigning order {level} "
            f"(volume={current_order.volume})"
        )
        print(f"  Partial assignment: {assignment}")

        for batch_id in existing_batches:
            batch_load = sum(
                solver.orders[index].volume
                for index, assigned_batch in assignment.items()
                if assigned_batch == batch_id
            )
            new_load = batch_load + current_order.volume

            if new_load > solver.capacity:
                print(
                    f"  Reject batch {batch_id}: load {batch_load} + "
                    f"{current_order.volume} = {new_load} exceeds capacity "
                    f"{solver.capacity}."
                )
                continue

            print(
                f"  Try batch {batch_id}: load {batch_load} + "
                f"{current_order.volume} = {new_load}; capacity is respected."
            )
            assignment[level] = batch_id
            remaining = range(level + 1, len(solver.orders))
            if solver.propagate_constraints(assignment, remaining):
                print("    Forward check passed.")
                trace_search(assignment, level + 1)
            else:
                print("    Forward check rejected this branch.")
            del assignment[level]

        new_batch_id = max(existing_batches) + 1 if existing_batches else 0
        if current_order.volume <= solver.capacity:
            print(
                f"  Try new batch {new_batch_id}: load 0 + "
                f"{current_order.volume} = {current_order.volume}."
            )
            assignment[level] = new_batch_id
            remaining = range(level + 1, len(solver.orders))
            if solver.propagate_constraints(assignment, remaining):
                print("    Forward check passed.")
                trace_search(assignment, level + 1)
            else:
                print("    Forward check rejected this branch.")
            del assignment[level]
        else:
            print(
                f"  Reject new batch: order volume {current_order.volume} "
                f"exceeds capacity {solver.capacity}."
            )

    trace_search({}, 0)


demonstrate_search_tree(solver, max_depth=4)

## 5. Examine how search effort scales

The following experiment creates reproducible random instances at several sizes. The seed depends on the instance size, as shown in the code. Each instance uses the same solver, capacity, and time limit. Search time and node counts are empirical measurements; they are not a proof of asymptotic performance. The time limit also means a reported cost may be only the best cost found so far.

In [ ]:
def scalability_analysis():
    """Measure search effort for reproducibly generated small instances."""
    print("Scalability analysis")
    test_sizes = [4, 5, 6, 7, 8]
    experiment_rows = []

    for size in test_sizes:
        np.random.seed(42 + size)
        generated_orders = []
        for order_index in range(size):
            volume = np.random.randint(1, 5)
            location = (
                np.random.randint(1, 10),
                np.random.randint(1, 10),
            )
            generated_orders.append(Order(order_index, volume, location))

        experiment_solver = BacktrackingSolver(generated_orders, capacity=8)
        result = experiment_solver.solve(max_time=10)
        experiment_rows.append(
            {
                "size": size,
                "time_seconds": result["search_time"],
                "nodes_explored": result["nodes_explored"],
                "branches_rejected": result["pruned_nodes"],
                "rejection_rate_percent": result["prune_percentage"],
                "best_estimated_cost": result["cost"],
                "status": result["status"],
            }
        )
        print(
            f"Size {size}: {result['search_time']:.4f} seconds, "
            f"{result['nodes_explored']:,} states explored, "
            f"{result['pruned_nodes']:,} branches rejected, "
            f"status: {result['status']}"
        )

    return pd.DataFrame(experiment_rows)


scalability_df = scalability_analysis()
print("\nScalability results:")
print(scalability_df.round(4).to_string(index=False))

## 6. Visualize search and scaling results

The panels summarize search events, the proportion of explored and rejected events, observed runtime across the generated instance sizes, and the operational rejection rate across those sizes. These plots describe this experiment; they should not be interpreted as universal performance guarantees.

In [ ]:
def create_visualizations(results, scalability_df):
    """Plot search statistics and empirical scaling measurements."""
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle(
        "Order Batching: Constraint-Propagation Backtracking",
        fontsize=16,
        fontweight="bold",
    )

    ax1 = axes[0, 0]
    statistics = [
        "States explored",
        "Branches rejected",
        "Backtracking calls",
        "Forward-check calls",
    ]
    values = [
        results["nodes_explored"],
        results["pruned_nodes"],
        results["backtrack_calls"],
        results["propagation_calls"],
    ]
    colors = ["skyblue", "lightcoral", "lightgreen", "gold"]
    bars = ax1.bar(statistics, values, color=colors, alpha=0.85)
    ax1.set_ylabel("Count")
    ax1.set_title("Search statistics")
    ax1.tick_params(axis="x", rotation=30)
    for bar, value in zip(bars, values):
        ax1.annotate(
            f"{value:,}",
            (bar.get_x() + bar.get_width() / 2, bar.get_height()),
            xytext=(0, 3),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=9,
        )

    ax2 = axes[0, 1]
    event_counts = [results["nodes_explored"], results["pruned_nodes"]]
    if sum(event_counts) > 0:
        ax2.pie(
            event_counts,
            labels=["States explored", "Branches rejected"],
            colors=["lightblue", "salmon"],
            autopct="%1.1f%%",
            startangle=90,
        )
    else:
        ax2.text(0.5, 0.5, "No search events to display", ha="center", va="center")
    ax2.set_title(
        f"Explored and rejected events\n"
        f"Operational rejection rate: {results['prune_percentage']:.1f}%"
    )

    ax3 = axes[1, 0]
    ax3.plot(
        scalability_df["size"],
        scalability_df["time_seconds"],
        "o-",
        label="Search time",
        linewidth=2,
        markersize=8,
        color="blue",
    )
    ax3.set_xlabel("Problem size (orders)")
    ax3.set_ylabel("Time (seconds)")
    ax3.set_title("Observed search time by problem size")
    ax3.legend()
    ax3.grid(True, alpha=0.3)

    ax4 = axes[1, 1]
    ax4.plot(
        scalability_df["size"],
        scalability_df["rejection_rate_percent"],
        "s-",
        label="Operational rejection rate",
        linewidth=2,
        markersize=8,
        color="red",
    )
    ax4.set_xlabel("Problem size (orders)")
    ax4.set_ylabel("Rejected events (%)")
    ax4.set_title("Observed rejection rate by problem size")
    ax4.legend()
    ax4.grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()
    return fig


visualization_fig = create_visualizations(results, scalability_df)
print("Visualizations created successfully.")

## 7. Interpret complexity and guarantees

The recursion depth is the number of orders, $n$. At a state with $k$ existing batches, the next order can be tried in up to $k$ existing batches and one new batch, subject to capacity. Canonical batch labels avoid exploring equivalent label permutations, but the number of distinct partitions can still grow as the Bell number $B_n$. Evaluating one complete assignment also computes a nearest-neighbor route for each batch; across all batches this takes at most quadratic work in the number of orders. A useful worst-case upper bound is therefore $O(B_n n^2)$ time for this implementation, with $O(n)$ recursion and assignment storage, in addition to the $O(n^2)$ distance matrix.

The search is exact with respect to the stated batch-cost estimate only when it finishes. The nearest-neighbor route estimate is itself a heuristic, and the forward check is weak when unlimited new batches are allowed. Capacity checks still reject assignments that exceed capacity. A time-limited run provides an incumbent solution if one was found, but does not establish optimality.

In [ ]:
print("Algorithm complexity and run interpretation")
print("\nSearch structure:")
print("- Depth: one assignment level per order.")
print("- Candidate choices: feasible existing batches plus one new batch.")
print("- Number of canonical partitions: bounded by the Bell number B_n.")
print("- Worst-case evaluation bound: O(B_n * n^2).")
print("- Assignment and recursion storage: O(n), apart from the O(n^2) distance matrix.")

print("\nRun-specific measurements:")
print(f"- Search status: {results['status']}")
print(f"- States explored: {results['nodes_explored']:,}")
print(f"- Branches rejected: {results['pruned_nodes']:,}")
print(f"- Forward-check calls: {results['propagation_calls']:,}")
print(f"- Search time: {results['search_time']:.4f} seconds")
if results["solution"] is not None:
    print(f"- Best estimated travel cost: {results['cost']:.2f}")
else:
    print("- No feasible complete assignment was found.")

print("\nInterpretation:")
if results["status"] == "complete" and results["solution"] is not None:
    print("- The search completed and found the minimum cost for the stated model.")
elif results["solution"] is not None:
    print("- A feasible incumbent was found, but optimality is not established.")
else:
    print("- The run did not find a feasible complete assignment.")
print("- The route-cost estimate uses nearest-neighbor tours, not exact route optimization.")

## 8. Summary and conclusions

This notebook formulates order batching as a capacity-constrained assignment problem and solves it with recursive depth-first backtracking. It uses forward checking to test whether unassigned orders still have a feasible batch, while capacity checks reject invalid assignments as the search branches. The best complete assignment is selected by the sum of nearest-neighbor depot-to-depot route estimates.

Interpret the result in two stages: first check whether the search status is `complete`; then interpret the reported cost as optimal for the model only if the search completed. Regardless of completion, the route-cost estimate is based on a nearest-neighbor heuristic, not an exact route optimizer. The approach is most useful for understanding systematic search and for small instances where exhaustive exploration is practical. Its main limitations are combinatorial growth, the weak forward check under unlimited batch creation, and the simplifying assumptions about warehouse travel.